In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
import os
from catboost import CatBoostRegressor


warnings.filterwarnings('ignore')


In [ ]:
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Read the dataset

df = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

In [ ]:
# Task 2: Write your code here:

print(df.head())

In [ ]:
# Task 3: Write your code here:

print(df.info())

In [ ]:
# Task 4: Write your code here:

print(df.describe())

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(10, 6))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.title('Distribution of Delivery Time')
plt.show()

In [ ]:
# Task 1: Write your code here:

df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:

# Drop rows
df = df.dropna(subset=['Delivery_Time'])

# Fill missing values
df = df.dropna(subset=['Delivery_Time'])
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median(), inplace=True)
df['Weather'].fillna(df['Weather'].mode()[0], inplace=True)
df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0], inplace=True)
df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0], inplace=True)

In [ ]:
# Task 3: Write your code here:

df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:

df = pd.get_dummies(df, columns=['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'], drop_first=True)

In [ ]:
# Task 5: Write your code here:

X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
# Task 6: Write your code here:

print(" :) ")

In [ ]:
# Task 1: Write your code here:

print(f"X shape: {X_scaled.shape}, y shape: {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_scaled), 1):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)
    print(f"Fold {fold} - MAE: {mae:.4f}")

avg_mae = np.mean(mae_scores)
print(f"Average MAE: {avg_mae:.4f}")

final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X_scaled, y)
final_predictions = final_model.predict(X_scaled)

In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': X_scaled.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 6))
plt.hist(final_predictions, bins=30, edgecolor='black', color='orange')
plt.xlabel('Predicted Delivery Time')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Delivery Time')
plt.show()

In [ ]:
# Task Bonus: Write your code here:


scores = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_idx, val_idx) in enumerate(kf.split(X_scaled)):
    X_train = X_scaled.iloc[train_idx]
    X_val = X_scaled.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # first
    model1 = RandomForestRegressor(n_estimators=100, random_state=42)
    model1.fit(X_train, y_train)
    pred1 = model1.predict(X_val)
    # scond
    model2 = CatBoostRegressor(iterations=100, random_state=42, verbose=0)
    model2.fit(X_train, y_train)
    pred2 = model2.predict(X_val)

    # combine predictions
    final_pred = (pred1 + pred2) / 2

    # calculate error
    error = mean_absolute_error(y_val, final_pred)
    scores.append(error)
    print(f"Fold {i+1} MAE: {error:.4f}")

print(f"\nMean MAE: {np.mean(scores):.4f}")